# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a FAIR^2 dataset using the `mlcroissant` library, referencing data entities by their `@id`. 

### Dataset Source
This dataset is defined by a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Install the mlcroissant library if not present
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their `@id`, including fields/columns with their `@id` references.

This helps specify which record sets and fields to extract for further analysis.

In [ ]:
# Examine the record sets in the dataset
record_sets = list(dataset.record_sets)
print("RecordSets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    # Print fields and their @ids
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - @id: {field['@id']} Name: {field.get('name', '')} Description: {field.get('description', '')}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`.
Use the identified record sets from above for extraction.

In [ ]:
# Collect record set @ids to extract

# Example usage: You may need to update this list depending on available record sets.
record_set_ids = [rs['@id'] for rs in record_sets]
print("RecordSet @ids for extraction:")
for r in record_set_ids:
    print(r)

# Extract and load all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nDataFrame for record_set '{record_set_id}': Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not extract records for record_set '{record_set_id}': {e}")

## 4. Exploratory Data Analysis (EDA)
Process a record set in detail. 
We will select a numeric field (e.g., age at diagnosis, hormone levels) using their field/column `@id`. 
Perform filtering, normalization, and grouping, referencing entities by `@id` throughout.

In [ ]:
# Choose a record set to process, e.g. Table 1, Table 2, or Table 3. Adjust below accordingly.
if record_set_ids:
    record_set_id = record_set_ids[0]  # First record set, update as needed
    df = dataframes.get(record_set_id)
    print(f"Processing DataFrame for record_set @id '{record_set_id}'")
    print(df.head())
    
    # Identify a numeric field for demo
    numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Replace with the actual field @id as needed
        print(f"Selected numeric field @id: {numeric_field_id}")

        # Filtering step: show records above a threshold
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field (choose one, replace as needed)
        group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in this record set.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize numeric fields and relationships between fields in the selected record set.

In [ ]:
# Example visualization: Distribution and relationship
if record_set_ids and df is not None and numeric_fields:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id is defined:
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing a FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id`. 

- Dataset meta and tabular content is accessible using Croissant schema and the `mlcroissant` API.
- Record sets, fields, and columns can be referenced and extracted by `@id` for robust and reproducible workflows.
- Basic data analysis, filtering, normalization, and visualization can be performed directly in pandas and matplotlib/seaborn.

Further analysis can focus on specific clinical, hormone, or genetic fields as needed, using their `@id` as reference.